# CircuitVision • Sam `data.zip` Dataset Validation

This **CPU-only** notebook checks the exact `data.zip` supplied from Sam's YOLO project. It does **not** train, install, delete, merge or upload any images to GitHub.

It verifies archive contents, source class names, actual image/YOLO label pairing, bounding-box formats, split layout, and any obvious class-ID errors **before using GPU time**. The result is a small `Sam_Dataset_Audit_Report.zip` to upload in ChatGPT.

Expected Sam model class order: `BUZZER`, `CERAMIC CAPACITOR`, `DIODE`, `LED (BLUE)`, `LED (GREEN)`, `POTENTIOMETER`, `POWER MODULE`, `RESISTOR 10K`, `RESISTOR 2K`, `RESISTOR 52K`.

**This audit does not prove labels are visually correct or that resistor values are measured.**

## 1. Select the 173 MB ZIP on Android, or find it in Google Drive

On a phone, `files.upload()` opens the file chooser. If the original ZIP is in Drive, mount Drive to avoid uploading it again; the notebook checks the original `MyDrive/YOLO/dataset/data.zip` path first.

In [ ]:
from pathlib import Path
import json,zipfile,hashlib,collections,os,re,random,io,math,stat
SOURCE=Path('/content/drive/MyDrive/YOLO/dataset/data.zip')
if not SOURCE.is_file():
    from google.colab import files,drive
    print('Looking for original Sam data.zip on your Google Drive first.')
    try:
        drive.mount('/content/drive')
    except Exception as exc:
        print('Drive mount not completed:',str(exc)[:200])
    SOURCE=Path('/content/drive/MyDrive/YOLO/dataset/data.zip')
    if not SOURCE.is_file():
        print('Sam data.zip not found at the original Drive path. Upload data.zip now.')
        uploaded=files.upload()
        if len(uploaded)==1:
            SOURCE=Path('/content')/next(iter(uploaded))
assert SOURCE.is_file(),'Cannot find Sam annotated data.zip'
assert zipfile.is_zipfile(SOURCE),'This is not a valid ZIP archive'
size=SOURCE.stat().st_size
print('ZIP:',SOURCE.name,'size:',size,'bytes')
print('SHA256:',hashlib.file_digest(open(SOURCE,'rb'),'sha256').hexdigest())

## 2. Inspect file structure and metadata safely (without extracting)

In [ ]:
from pathlib import PurePosixPath
IMAGE_EXTS={'.jpg','.jpeg','.png','.webp','.bmp'}
EXPECTED=['BUZZER','CERAMIC CAPACITOR','DIODE','LED (BLUE)','LED (GREEN)',
          'POTENTIOMETER','POWER MODULE','RESISTOR 10K','RESISTOR 2K','RESISTOR 52K']
def normalize(s):
    return re.sub(r'\s+',' ',re.sub(r'[_-]+',' ',str(s).strip().upper())).strip()
with zipfile.ZipFile(SOURCE) as z:
    files_in_zip=[x for x in z.infolist() if not x.is_dir()]
    allnames=set(x.filename for x in files_in_zip)
    badpaths=[x.filename for x in files_in_zip if PurePosixPath(x.filename).is_absolute()
              or '..' in PurePosixPath(x.filename).parts or '\\' in x.filename]
    assert not badpaths, 'Unsafe archive paths detected: '+repr(badpaths[:5])
    summary=dict(files=len(files_in_zip),uncompressed_bytes=sum(x.file_size for x in files_in_zip),
                 sample_paths=[x.filename for x in files_in_zip[:35]],
                 formats=dict(collections.Counter(PurePosixPath(x.filename).suffix.lower() for x in files_in_zip)))
    namesfiles=[x.filename for x in files_in_zip if PurePosixPath(x.filename).name.lower() in
                ('classes.txt','data.yaml','dataset.yaml','obj.names','data.names')]
    summary['class_mapping_file_candidates']=namesfiles
    summary['class_mapping_raw']={}
    for name in namesfiles[:10]:
        summary['class_mapping_raw'][name]=z.read(name).decode('utf-8-sig',errors='replace')[:6000]
    image_files=[x.filename for x in files_in_zip if PurePosixPath(x.filename).suffix.lower() in IMAGE_EXTS]
    annot_files=[x.filename for x in files_in_zip if PurePosixPath(x.filename).suffix.lower()=='.txt' and
                 PurePosixPath(x.filename).name.lower() not in ('classes.txt','train.txt','val.txt','valid.txt','test.txt','obj.names','data.names')]
    summary['image_count']=len(image_files)
    summary['annotation_text_files']=len(annot_files)
print('Archive paths (sample):')
print('\n'.join(summary['sample_paths'][:20]))
print('Image count:',summary['image_count'],'YOLO label candidate count:',summary['annotation_text_files'])
print('Class definition files:',namesfiles)
for path,content in summary['class_mapping_raw'].items():
    print('\n',path,'\n',content[:500])

## 3. Check class order, YOLO box coordinates, and image/label pairs

In [ ]:
import yaml
with zipfile.ZipFile(SOURCE) as z:
    mapping=None
    if namesfiles:
        for name in namesfiles:
            raw=summary['class_mapping_raw'].get(name,'')
            leaf=PurePosixPath(name).name.lower()
            if leaf in ('classes.txt','obj.names','data.names'):
                candidate=[s.strip() for s in raw.splitlines() if s.strip()]
            else:
                try:
                    parsed=yaml.safe_load(raw)
                    classes=parsed.get('names',[]) if isinstance(parsed,dict) else []
                    candidate=([classes[k] for k in sorted(classes,key=lambda k:int(k))]
                               if isinstance(classes,dict) else classes)
                except Exception: candidate=[]
            if len(candidate)==10:
                mapping=[str(v) for v in candidate]; break
    if mapping is not None:
        summary['class_names']=mapping
        summary['class_mapping_matches_sam']=([normalize(s) for s in mapping]==
                                             [normalize(s) for s in EXPECTED])
    else:
        summary['class_names']=None;summary['class_mapping_matches_sam']=False
    all_labels=set(annot_files)
    matched=0;missing=0;matched_paths=[]
    class_counts=collections.Counter()
    parse_errors=[]
    file_boxes=0
    for image_path in image_files:
        p=PurePosixPath(image_path)
        candidate1=str(p.with_suffix('.txt'))
        candidate2=str(p.parent.parent/'labels'/(p.stem+'.txt')) if p.parent.name=='images' else ''
        label_path=next((x for x in (candidate1,candidate2) if x in all_labels),None)
        if label_path is None:
            missing+=1;continue
        matched+=1
        if len(matched_paths)<30:matched_paths.append({'image':image_path,'label':label_path})
        try:
            text=z.read(label_path).decode('utf-8-sig')
            for line in text.splitlines():
                if not line.strip():continue
                vals=line.split()
                if len(vals)!=5 and not (len(vals)>=7 and len(vals)%2==1):
                    raise ValueError('expected YOLO bbox or polygon row, got '+str(len(vals))+' fields')
                cls=int(vals[0]); coords=[float(v) for v in vals[1:]]
                if not (0<=cls<10) or any(not math.isfinite(v) or not 0<=v<=1 for v in coords):
                    raise ValueError('out-of-range YOLO class or coordinates')
                if len(coords)==4 and (coords[2]==0 or coords[3]==0):
                    raise ValueError('zero-size box')
                class_counts[str(cls)]+=1;file_boxes+=1
        except Exception as e:
            if len(parse_errors)<40:parse_errors.append({'label':label_path,'problem':str(e)[:300]})
    summary['image_label_pairs']=matched
    summary['images_missing_labels']=missing
    summary['box_counts_by_id']=dict(class_counts)
    summary['total_boxes']=file_boxes
    summary['parse_error_samples']=parse_errors
    summary['matched_paths_sample']=matched_paths
    summary['same_image_duplicates_not_checked']=True
    summary['can_merge_automatically']=(bool(summary['class_mapping_matches_sam'])
        and matched>=30 and not parse_errors and all(class_counts.get(str(i),0)>0 for i in range(10)))
print('Matching class names:',summary['class_mapping_matches_sam'])
print('Image/label pairs:',matched,'unpaired images:',missing)
print('Annotated boxes per class ID:',dict(class_counts))
print('Annotation errors (sample):',parse_errors[:5])
print('Safe automatic merge preflight:',summary['can_merge_automatically'])
if not summary['can_merge_automatically']:
    print('STOP: We must adjust the unified training notebook to the actual ZIP structure before GPU training.')

## 4. Download only the compact audit JSON

Send the report ZIP to Rad in this chat. **Do not run training yet** if the audit shows missing classes, mismatched labels or invalid annotation formats.

In [ ]:
folder=Path('/content/Sam_Dataset_Audit_Report.json')
folder.write_text(json.dumps({'dataset':'Sam original YOLO data.zip','audit':summary,
                              'note':'No visual label accuracy or data leakage assessment has been performed.'},indent=2))
output=Path('/content/Sam_Dataset_Audit_Report.zip')
with zipfile.ZipFile(output,'w',zipfile.ZIP_DEFLATED) as z:z.write(folder,folder.name)
print('Ready:',output,output.stat().st_size,'bytes')
from google.colab import files
files.download(str(output))